In [20]:
import sys
print(sys.executable)

/Users/hovietbach/miniforge3/envs/Financial_Fraud_Detection_Thesis/bin/python


In [21]:
import os
print(os.environ.get("CONDA_DEFAULT_ENV"))

Financial_Fraud_Detection_Thesis


# LOAD DATA
We will load all the data except 219 V columns that were determined redundant by correlation analysis

In [22]:
BUILD95 = True
BUILD96 = True

import numpy as np, pandas as pd, os, gc
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:
train_transaction = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/train_transaction.csv')

train_identity = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/train_identity.csv')

test_transaction = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/test_transaction.csv')

test_identity = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/test_identity.csv')

In [ ]:
train_transaction

In [ ]:
test_transaction

In [ ]:
print(max(train_transaction.TransactionDT)/(24*60*60)) # The last day in the train_transaction

In [ ]:
train_identity

In [ ]:
string_cols_transaction = train_transaction.select_dtypes(include=["object", "string"]).columns.tolist()
print(string_cols_transaction)

In [ ]:
string_cols_identity = train_identity.select_dtypes(include=["object", "string"]).columns.tolist()
print(string_cols_identity)

In [ ]:
string_cols_test_identity = [
    c for c in test_identity.select_dtypes(include=["object", "string"]).columns
    if c not in ["DeviceType", "DeviceInfo"]
]
print(string_cols_test_identity)

In [ ]:
str_type = string_cols_transaction + string_cols_identity + string_cols_test_identity
print(str_type)
print(len(str_type))

In [ ]:
## FIRST 53 COLUMNS
cols = train_transaction.columns[
    ~train_transaction.columns.str.startswith("V") &
    (train_transaction.columns != "isFraud")
].tolist()
print(cols)

In [ ]:
# Number of V columns
v_cols = train_transaction.columns[
    train_transaction.columns.str.startswith("V")
].tolist()
print(len(v_cols))

In [ ]:
# V COLUMNS TO LOAD DECIDED BY CORRELATION EDA
# https://www.kaggle.com/cdeotte/eda-for-columns-v-and-id

v =  [1, 3, 4, 6, 8, 11]
v += [13, 14, 17, 20, 23, 26, 27, 30]
v += [36, 37, 40, 41, 44, 47, 48]
v += [54, 56, 59, 62, 65, 67, 68, 70]
v += [76, 78, 80, 82, 86, 88, 89, 91]

#v += [96, 98, 99, 104] #relates to groups, no NAN
v += [107, 108, 111, 115, 117, 120, 121, 123] # maybe group, no NAN
v += [124, 127, 129, 130, 136] # relates to groups, no NAN

# LOTS OF NAN BELOW
v += [138, 139, 142, 147, 156, 162] #b1
v += [165, 160, 166] #b1
v += [178, 176, 173, 182] #b2
v += [187, 203, 205, 207, 215] #b2
v += [169, 171, 175, 180, 185, 188, 198, 210, 209] #b2
v += [218, 223, 224, 226, 228, 229, 235] #b3
v += [240, 258, 257, 253, 252, 260, 261] #b3
v += [264, 266, 267, 274, 277] #b3
v += [220, 221, 234, 238, 250, 271] #b3

v += [294, 284, 285, 286, 291, 297] # relates to groups, no NAN
v += [303, 305, 307, 309, 310, 320] # relates to groups, no NAN
v += [281, 283, 289, 296, 301, 314] # relates to groups, no NAN
#v += [332, 325, 335, 338] # b4 lots NAN

cols += ['V'+str(x) for x in v]
dtypes = {}
for c in cols+['id_0'+str(x) for x in range(1,10)]+['id_'+str(x) for x in range(10,34)]+\
    ['id-0'+str(x) for x in range(1,10)]+['id-'+str(x) for x in range(10,34)]:
        dtypes[c] = 'float32'
for c in str_type: dtypes[c] = 'category'

In [ ]:
print("Number of kept V-columns: ", len(v))

In [ ]:
print('Number of eliminated V-columns: ', len(v_cols) - len(v))

In [ ]:
print(dtypes)

In [ ]:
print(cols)

In [ ]:
%%time
# LOAD TRAIN
X_train = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/train_transaction.csv', index_col='TransactionID', dtype=dtypes, usecols=cols + ['isFraud'])

In [ ]:
train_transaction

In [ ]:
X_train

In [ ]:
train_id = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/train_identity.csv', index_col='TransactionID', dtype=dtypes)

In [ ]:
train_id

In [ ]:
print(train_transaction.dtypes.to_frame("dtype"))
print(train_identity.dtypes.to_frame("dtype"))

In [ ]:
print(X_train.dtypes.to_frame("dtype"))
print(train_id.dtypes.to_frame("dtype"))

In [23]:
X_train = X_train.merge(train_id, how='left', left_index=True, right_index=True)
print(X_train)
print(X_train.dtypes)

NameError: name 'X_train' is not defined

In [ ]:
# LOAD TEST
X_test = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/test_transaction.csv', index_col='TransactionID', dtype=dtypes, usecols=cols)
test_id = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_imported_file/test_identity.csv', index_col='TransactionID', dtype=dtypes)
fix = {o:n for o, n in zip(test_id.columns, train_id.columns)}
print(fix)
# print("test_id before renaming: ", test_id)
test_id.rename(columns=fix, inplace=True)
X_test = X_test.merge(test_id, how='left', left_index=True, right_index=True)
# TARGET
y_train = X_train['isFraud'].copy()
# del train_id, test_id, X_train['isFraud']; x = gc.collect()
del X_train['isFraud']; x = gc.collect()
# PRINT STATUS
print('Train shape',X_train.shape,'test shape',X_test.shape)

In [ ]:
X_test

In [ ]:
test_id

In [ ]:
print(X_test.dtypes.to_frame("dtype"))
print(test_id.dtypes.to_frame("dtype"))

In [ ]:
X_train

In [ ]:
d_cols = X_train.columns[
    X_train.columns.str.startswith("D")
].tolist()
d_cols.remove('DeviceType')
d_cols.remove('DeviceInfo')
print(d_cols)
print(len(d_cols))

# NORMALIZE D COLUMNS
The D Columns are "time deltas" from some point in the past. We will transform the D Columns into their point in the past. This will stop the D columns from increasing with time. The formula is D15n = Transaction_Day - D15 and Transaction_Day = TransactionDT/(24*60*60). Afterward we multiply this number by negative one.

In [ ]:
# PLOT ORIGINAL D
plt.figure(figsize=(15,5))
plt.scatter(X_train.TransactionDT,X_train.D15)
plt.title('Original D15')
plt.xlabel('Time')
plt.ylabel('D15')
plt.show()

In [ ]:
# NORMALIZE D COLUMNS
# for i in range(1,16):
#     if i in [1,2,3,5,9]: continue  # Why?
#     X_train['D'+str(i)] =  X_train['D'+str(i)] - X_train.TransactionDT/np.float32(24*60*60)
#     X_test['D'+str(i)] = X_test['D'+str(i)] - X_test.TransactionDT/np.float32(24*60*60)

for i in range(1, 16):
    if i in [1, 2, 3, 5, 9]:
        continue
    c = f"D{i}"

    X_train[c] = X_train[c] - np.floor(X_train["TransactionDT"] / np.float32(24 * 60 * 60))
    X_test[c]  = X_test[c]  - np.floor(X_test["TransactionDT"]  / np.float32(24 * 60 * 60))


In [ ]:
X_train

In [ ]:
X_test

In [ ]:
# PLOT TRANSFORMED D
plt.figure(figsize=(15,5))
plt.scatter(X_train.TransactionDT,X_train.D15)
plt.title('Transformed D15')
plt.xlabel('Time')
plt.ylabel('D15n')
plt.show()

In [ ]:
numeric_cols = list(set(cols) - set(str_type))
print(numeric_cols)

In [ ]:
print("X_train.dtypes:\n", X_train.dtypes)
print("X_test.dtypes:\n", X_test.dtypes)

In [ ]:
%%time
# LABEL ENCODE AND MEMORY REDUCE
X_train_copy1 = X_train.copy()
X_test_copy1 = X_test.copy()
for f in X_train_copy1.columns:
    if str(X_train_copy1[f].dtype) == "category" or X_train_copy1[f].dtype == "object":
        comb, _ = pd.factorize(pd.concat([X_train_copy1[f], X_test_copy1[f]]), sort=True)
        X_train_copy1[f] = comb[:len(X_train_copy1)].astype("int16")
        X_test_copy1[f]  = comb[len(X_train_copy1):].astype("int16")
    elif f not in ["TransactionAmt", "TransactionDT"]:
        mn = np.min((X_train_copy1[f].min(), X_test_copy1[f].min()))
        X_train_copy1[f] = (X_train_copy1[f] - np.float32(mn)).fillna(-1)
        # X_train_copy1[f + "_isna"] = X_train_copy1[f].isna().astype("int8")

        X_test_copy1[f]  = (X_test_copy1[f]  - np.float32(mn)).fillna(-1)
        # X_test_copy1[f + "_isna"] = X_test_copy1[f].isna().astype("int8")

In [ ]:
print(str_type)

In [ ]:
print(numeric_cols)

In [ ]:
print(numeric_cols)
# keep only columns that actually exist in X_train
num_cols = [c for c in numeric_cols if c in X_train.columns]

# NaN count per numeric column
na_counts = X_train[num_cols].isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)

print(na_counts)

In [ ]:
# keep only columns that actually exist in X_train_copy1
num_cols = [c for c in numeric_cols if c in X_train_copy1.columns]

# NaN count per numeric column
na_counts = X_train_copy1[num_cols].isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)

print(na_counts)

In [ ]:
# Check column that remains NaN
print(X_train.isna().sum()[X_train.isna().sum() > 0])
print(X_test.isna().sum()[X_test.isna().sum() > 0])

In [ ]:
# Check column that remains NaN
print(X_train_copy1.isna().sum()[X_train_copy1.isna().sum() > 0])
print(X_test_copy1.isna().sum()[X_test_copy1.isna().sum() > 0])

In [ ]:
print("X_train_copy.dtypes:\n", X_train.dtypes)
print("X_test_copy.dtypes:\n", X_test.dtypes)

In [ ]:
print("X_train_copy1.dtypes:\n", X_train_copy1.dtypes)
print("X_test_copy1.dtypes:\n", X_test_copy1.dtypes)

In [ ]:
d_cols = X_train_copy1.columns[X_train_copy1.columns.str.match(r"^D\d+$")]
print(X_train_copy1[d_cols].dtypes)

In [ ]:
X_train

In [ ]:
X_train_copy1

In [ ]:
X_test

In [ ]:
X_test_copy1

In [ ]:
X_train_copy2 = X_train_copy1.copy()
X_test_copy2 = X_test_copy1.copy()

In [ ]:
temp = X_train_copy1['card1'].value_counts().to_dict()
# X_train_copy1['card1_counts'] = X_train_copy1['card1'].map(temp)

In [ ]:
print(temp)

In [24]:
# FREQUENCY ENCODE TOGETHER
def encode_FE(df1, df2, cols):
    for col in cols:
        df = pd.concat([df1[col],df2[col]])
        vc = df.value_counts(dropna=True, normalize=False).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df1[nm] = df1[col].map(vc)
        df1[nm] = df1[nm].astype('float32')
        df2[nm] = df2[col].map(vc)
        df2[nm] = df2[nm].astype('float32')
        print(nm,', ',end='')

In [25]:
# LABEL ENCODE
# def encode_LE(col,train=X_train_copy1,test=X_test_copy1,verbose=True):
#     df_comb = pd.concat([train[col],test[col]],axis=0)
#     df_comb,_ = df_comb.factorize(sort=True)
#     nm = col
#     if df_comb.max()>32000:
#         train[nm] = df_comb[:len(train)].astype('int32')
#         test[nm] = df_comb[len(train):].astype('int32')
#     else:
#         train[nm] = df_comb[:len(train)].astype('int16')
#         test[nm] = df_comb[len(train):].astype('int16')
#     del df_comb; x=gc.collect()
#     if verbose: print(nm,', ',end='')

def encode_LE(col, train=X_train_copy2, test=X_test_copy2, verbose=True):
    df_comb = pd.concat([train[col], test[col]], axis=0)
    df_comb, _ = df_comb.factorize(sort=True)

    if df_comb.max() > 32000:
        train_vals = df_comb[:len(train)].astype("int32")
        test_vals = df_comb[len(train):].astype("int32")
    else:
        train_vals = df_comb[:len(train)].astype("int16")
        test_vals = df_comb[len(train):].astype("int16")

    train = _set_col_no_fragment(train, col, train_vals)
    test = _set_col_no_fragment(test, col, test_vals)

    del df_comb
    gc.collect()
    if verbose:
        print(col, ", ", end="")
    return train, test

NameError: name 'X_train_copy2' is not defined

In [ ]:
# COMBINE FEATURES
# def encode_CB(col1,col2,df1=X_train_copy1,df2=X_test_copy1):
#     nm = col1+'_'+col2
#     df1[nm] = df1[col1].astype(str)+'_'+df1[col2].astype(str)
#     df2[nm] = df2[col1].astype(str)+'_'+df2[col2].astype(str)
#     encode_LE(nm,verbose=False)
#     print(nm,', ',end='')


def encode_CB(col1, col2, df1=X_train_copy2, df2=X_test_copy2):
    nm = col1 + "_" + col2

    cb1 = df1[col1].astype(str) + "_" + df1[col2].astype(str)
    cb2 = df2[col1].astype(str) + "_" + df2[col2].astype(str)

    df1 = _set_col_no_fragment(df1, nm, cb1)
    df2 = _set_col_no_fragment(df2, nm, cb2)

    # keep original logic: CB -> LE
    df1, df2 = encode_LE(nm, train=df1, test=df2, verbose=False)
    print(nm, ", ", end="")
    return df1, df2

In [ ]:
def _set_col_no_fragment(df, col, values):
    s = pd.Series(values, index=df.index, name=col)
    if col in df.columns:
        df[col] = s.to_numpy()   # replace whole column; dtype can change
        return df
    return pd.concat([df, s], axis=1)

# ENCODING FUNCTIONS
Below are 5 encoding functions. (1) encode_FE does frequency encoding where it combines train and test first and then encodes. (2) encode_LE is a label encoded for categorical features (3) encode_AG makes aggregated features such as aggregated mean and std (4) encode_CB combines two columns (5) encode_AG2 makes aggregated features where it counts how many unique values of one feature is within a group. For more explanation about feature engineering, see the discussion

In [ ]:
# GROUP AGGREGATION NUNIQUE
def encode_AG2(main_columns, uids, train_df=X_train_copy2, test_df=X_test_copy2):
    for main_column in main_columns:
        for col in uids:
            comb = pd.concat([train_df[[col]+[main_column]],test_df[[col]+[main_column]]],axis=0)
            mp = comb.groupby(col)[main_column].agg(['nunique'])['nunique'].to_dict()
            train_df[col+'_'+main_column+'_ct'] = train_df[col].map(mp).astype('float32')
            test_df[col+'_'+main_column+'_ct'] = test_df[col].map(mp).astype('float32')
            print(col+'_'+main_column+'_ct, ',end='')

# Feature Engineering
We will now engineer features. All of these features where chosen because each increases local validation. The procedure for engineering features is as follows. First you think of an idea and create a new feature. Then you add it to your model and evaluate whether local validation AUC increases or decreases. If AUC increases keep the feature, otherwise discard the feature.

In [ ]:
train_new = pd.DataFrame({
    "cents": (X_train_copy2["TransactionAmt"] - np.floor(X_train_copy2["TransactionAmt"])).astype("float32"),
    "dollars": np.floor(X_train_copy2["TransactionAmt"]).astype("float32"),
}, index=X_train_copy2.index)

test_new = pd.DataFrame({
    "cents": (X_test_copy2["TransactionAmt"] - np.floor(X_test_copy2["TransactionAmt"])).astype("float32"),
    "dollars": np.floor(X_test_copy2["TransactionAmt"]).astype("float32"),
}, index=X_test_copy2.index)

X_train_copy2 = pd.concat([X_train_copy2, train_new], axis=1)
X_test_copy2 = pd.concat([X_test_copy2, test_new], axis=1)

In [ ]:
X_train_copy2

In [ ]:
X_test_copy2

In [ ]:
def encode_AG(main_columns, uids, aggregations=['mean'], train_df=X_train_copy2, test_df=X_test_copy2, fillna=True, usena=False):
    for main_column in main_columns:
        for col in uids:
            for agg_type in aggregations:
                new_col_name = f"{main_column}_{col}_{agg_type}"

                temp_df = pd.concat([train_df[[col, main_column]], test_df[[col, main_column]]], axis=0)
                if usena:
                    temp_df.loc[temp_df[main_column] == -1, main_column] = -1

                mp = temp_df.groupby(col, dropna=False)[main_column].agg(agg_type).to_dict()

                train_df[new_col_name] = train_df[col].map(mp).astype("float32")
                test_df[new_col_name]  = test_df[col].map(mp).astype("float32")

                if fillna:
                    train_df[new_col_name] = train_df[new_col_name].fillna(-1).astype("float32")
                    test_df[new_col_name]  = test_df[new_col_name].fillna(-1).astype("float32")
                print(f"'{new_col_name}', ", end="")

In [ ]:
print(X_train_copy2.dtypes)
print(X_test_copy2.dtypes)

In [ ]:
# FREQUENCY ENCODE: ADDR1, CARD1, CARD2, CARD3, P_EMAIL_DOMAIN
encode_FE(X_train_copy2,X_test_copy2,['addr1','card1','card2','card3','P_emaildomain'])

In [ ]:
# COMBINE COLUMNS CARD1+ADDR1, CARD1+ADDR1+P_EMAILDOMAIN
X_train_copy2, X_test_copy2 = encode_CB("card1", "addr1", X_train_copy2, X_test_copy2)

In [ ]:
X_train_copy2, X_test_copy2 = encode_CB('card1_addr1','P_emaildomain', X_train_copy2, X_test_copy2)

In [ ]:
# FREQUENCY ENCODE
encode_FE(X_train_copy2,X_test_copy2,['card1_addr1','card1_addr1_P_emaildomain'])

In [ ]:
print('card1_addr1' in X_train_copy2.columns, 'card1_addr1_P_emaildomain' in X_train_copy2.columns)
print('card1_addr1' in X_test_copy2.columns, 'card1_addr1_P_emaildomain' in X_test_copy2.columns)

In [ ]:
# GROUP AGGREGATE
encode_AG(
    ['TransactionAmt','D9','D11'],
    ['card1','card1_addr1','card1_addr1_P_emaildomain'],
    ['mean','std'],
    train_df=X_train_copy2,
    test_df=X_test_copy2,
    usena=True,
    fillna=False,
)

In [ ]:
print(X_train_copy2['card1_addr1'].dtypes)
print(X_train_copy2['card1_addr1_P_emaildomain'].dtypes)

In [ ]:
X_train_copy2

In [ ]:
X_test_copy2

In [ ]:
print(X_train_copy2.dtypes)

In [ ]:
# Check column that remains NaN
print(X_train_copy2.isna().sum()[X_train_copy2.isna().sum() > 0])
print(X_test_copy2.isna().sum()[X_test_copy2.isna().sum() > 0])

# INSPECT CREATION OF UIDs

In [ ]:
X_train_copy4 = X_train_copy2.copy()
X_test_copy4 = X_test_copy2.copy()

In [ ]:
# std columns: singleton groups -> fill 0 is common
std_cols = [c for c in X_train_copy4.columns if c.endswith("_std")]
for c in std_cols:
    X_train_copy4[c] = X_train_copy4[c].fillna(0)
    X_test_copy4[c]  = X_test_copy4[c].fillna(0)

# mean columns: fill with train global mean per base variable (example for D9/D11 means)
for c in [x for x in X_train_copy4.columns if x.endswith("_mean")]:
    fill_val = X_train_copy4[c].mean()
    X_train_copy4[c] = X_train_copy4[c].fillna(fill_val)
    # X_train_copy4[c + "_isna"] = X_train_copy4[c].isna().astype("int8")

    X_test_copy4[c]  = X_test_copy4[c].fillna(fill_val)
    # X_test_copy4[c + "_isna"]  = X_test_copy4[c].isna().astype("int8")

In [ ]:
# Check column that remains NaN
print(X_train_copy4.isna().sum()[X_train_copy4.isna().sum() > 0])
print(X_test_copy4.isna().sum()[X_test_copy4.isna().sum() > 0])

In [ ]:
X_train_copy4[['card1', 'addr1', 'card1_addr1']]

In [ ]:
X_train_copy4['addr1'].value_counts()

In [ ]:
X_train_copy4['isFraud'] = y_train

In [ ]:
X_train_copy4['day'] = np.floor(X_train_copy4.TransactionDT / (24*60*60))
X_train_copy4['uid'] = X_train_copy4.card1_addr1.astype(str)+'_'+np.floor(X_train_copy4.day-X_train_copy4.D1).astype(str)

X_test_copy4['day'] = np.floor(X_test_copy4.TransactionDT / (24*60*60))
X_test_copy4['uid'] = X_test_copy4.card1_addr1.astype(str)+'_'+np.floor(X_test_copy4.day-X_test_copy4.D1).astype(str)

In [ ]:
X_train_copy4

In [ ]:
X_test_copy4

In [ ]:
# Factorizing X_train_copy4, X_test_copy4
# X_train_copy4["uid"] = pd.factorize(X_train_copy4["uid"], sort=True)[0]
# X_test_copy4["uid"] = pd.factorize(X_test_copy4["uid"], sort=True)[0]

In [ ]:
# LOAD UID FROM KONSTATIN's NOTEBOOK
uid_v2 = pd.read_csv('/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/csv_exported_files/uids_part_1_v6.csv')

In [ ]:
uid_v2

In [ ]:
uid_lookup = uid_v2.set_index('TransactionID')[['uid']].rename(columns={'uid': 'uid_v2'})

In [ ]:
X_all_copy4 = pd.concat(
    [X_train_copy4, X_test_copy4],
    axis=0,
    ignore_index=False
)

In [ ]:
X_all_copy4['uid'] = pd.factorize(X_all_copy4["uid"], sort=True)[0]

In [ ]:
X_all_copy4 = X_all_copy4.merge(uid_lookup, left_index=True, right_index=True, how='left')

print(X_all_copy4.shape)
print(X_all_copy4[['uid', 'uid_v2']].head())

In [ ]:
X_all_copy4

In [ ]:
print(X_all_copy4['uid_v2'].isna().mean())

In [ ]:
print(X_all_copy4['uid_v2'].duplicated().any())
print(X_all_copy4['uid_v2'].duplicated().sum())

In [ ]:
uid_dup_rows = X_all_copy4[X_all_copy4['uid_v2'].duplicated(keep=False)].sort_values('uid_v2')
uid_dup_rows

In [26]:
train_idx = X_train_copy4.index
test_idx  = X_test_copy4.index

X_train_copy4 = X_all_copy4.loc[train_idx].copy()
X_test_copy4  = X_all_copy4.loc[test_idx].copy()

NameError: name 'X_train_copy4' is not defined

In [ ]:
X_train_copy4.shape

In [ ]:
X_test_copy4 = X_test_copy4.drop(['isFraud'], axis=1)

In [ ]:
X_test_copy4.shape

In [ ]:
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar
import datetime

START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
dates_range = pd.date_range("2017-10-01", "2019-01-01")
us_holidays = calendar().holidays(dates_range.min(), dates_range.max())

TIME_COLS = [
    "DT", "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday",
    "DT_M_total", "DT_W_total", "DT_D_total"
]

def add_time_features(df):
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop(columns=[c for c in TIME_COLS if c in df.columns], errors="ignore")

    dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
    iso_week = dt.dt.isocalendar().week.astype("int16")

    feats = pd.DataFrame({
        "DT": dt,
        "DT_M": (((dt.dt.year - 2017) * 12 + dt.dt.month)).astype("int8"),
        "DT_W": (((dt.dt.year - 2017) * 52 + iso_week)).astype("int16"),
        "DT_D": (((dt.dt.year - 2017) * 365 + dt.dt.dayofyear)).astype("int16"),
        "DT_hour": dt.dt.hour.astype("int8"),
        "DT_day_week": dt.dt.dayofweek.astype("int8"),
        "DT_day_month": dt.dt.day.astype("int8"),
        "DT_week_month": (((dt.dt.day - 1) // 7) + 1).astype("int8"),
        "is_december": (dt.dt.month == 12).astype("int8"),
        "is_holiday": dt.dt.normalize().isin(us_holidays).astype("int8"),
    }, index=df.index)

    return pd.concat([df, feats], axis=1)

X_train_copy4 = add_time_features(X_train_copy4)
X_test_copy4 = add_time_features(X_test_copy4)

for col in ["DT_M", "DT_W", "DT_D"]:
    vals = np.concatenate([X_train_copy4[col].to_numpy(), X_test_copy4[col].to_numpy()])
    fq = pd.Series(vals).value_counts().to_dict()
    X_train_copy4[f"{col}_total"] = X_train_copy4[col].map(fq)
    X_test_copy4[f"{col}_total"] = X_test_copy4[col].map(fq)

In [ ]:
encode_FE(X_train_copy4,X_test_copy4,['uid'])

In [ ]:
encode_FE(X_train_copy4,X_test_copy4,['uid_v2'])

In [ ]:
df_show = X_train_copy4.sort_values(["uid_FE", "uid"], ascending=[False, False])
display(df_show)

In [ ]:
print(f"Total number of uids: {df_show['uid'].nunique()}")

In [ ]:
df_show['uid_v2'].dtypes

In [ ]:
df_show['uid'].value_counts()

In [ ]:
# df_show[['uid']].astype({'uid': 'int64'}).to_csv(
#     '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/data/ieee-fraud-detection/csv_exported_files/uid.csv',
#     index=False
# )

In [ ]:
df_isFraud_mixed = df_show.groupby("uid").filter(lambda g: g["isFraud"].nunique() > 1)

In [ ]:
df_isFraud_mixed

In [ ]:
# mixed_isFraud_uid_count = list(df_isFraud_mixed["uid"].value_counts().items())
mixed_isFraud_uid_count = list(df_isFraud_mixed["uid"].value_counts().keys())

In [ ]:
print(len(mixed_isFraud_uid_count))
print(mixed_isFraud_uid_count)

In [ ]:
mixed_uids_rate = len(mixed_isFraud_uid_count) / df_show['uid'].nunique()

mixed_rows_rate = len(df_isFraud_mixed) / len(df_show)

print(mixed_uids_rate)
print(mixed_rows_rate)

In [ ]:
# Filter to take the df with 'isFraud' == 0/1 and not isFraud mixed
mask = (
    X_train_copy4["isFraud"].isin([0, 1]) &
    (~X_train_copy4["uid_v2"].isin(mixed_isFraud_uid_count))
)

df_show_filtered = (
    X_train_copy4.loc[mask]
    .sort_values(["uid_v2_FE", "uid_v2"], ascending=[False, False])
)

In [ ]:
df_show_filtered['isFraud'].value_counts()

In [ ]:
print(len(df_show_filtered["uid_v2"].value_counts()))

# 3 groups with largest number of same 'uid'
df_show_filtered["uid_v2"].value_counts().head(5)

In [ ]:
topic_uids_filtered = df_show_filtered["uid_v2"].value_counts().head(3).index
print(topic_uids_filtered)

In [ ]:
top_uids = df_show["uid_v2"].value_counts().head(3).index
print(top_uids)

In [ ]:
uid_v2 = 3428022.0

In [ ]:
print(uid_v2)
train_cnt = (X_train_copy4["uid_v2"] == uid_v2).sum()
test_cnt  = (X_test_copy4["uid_v2"] == uid_v2).sum()
show_cnt  = (df_show_filtered["uid_v2"] == uid_v2).sum()
fe_val    = X_train_copy4.loc[X_train_copy4["uid_v2"] == uid_v2, "uid_v2_FE"].iloc[0]

print("train_cnt:", train_cnt)
print("test_cnt :", test_cnt)
print("show_cnt :", show_cnt)
print("uid_FE   :", fe_val)          # should match train_cnt + test_cnt
print("train+test:", train_cnt + test_cnt)

In [ ]:
def plot_uid_timeseries(df, uid_value, isfraud="all", start_date=None, uid_col="uid"):
    """
    uid_value: int/str uid to filter
    isfraud: 0, 1, or "all"
    start_date: required only if df has no DT column (uses TransactionDT + start_date)
    """
    d = df.copy()

    # Build DT if missing
    if "DT" not in d.columns:
        if "TransactionDT" not in d.columns or start_date is None:
            raise ValueError("Need 'DT' column, or provide start_date with 'TransactionDT'.")
        d["DT"] = pd.to_datetime(start_date) + pd.to_timedelta(d["TransactionDT"], unit="s")

    # Robust uid matching (works for int/str uid formats)
    uid_mask = d[uid_col].astype(str) == str(uid_value)

    if isfraud in (0, 1):
        mask = uid_mask & (d["isFraud"] == isfraud)
    elif isfraud == "all":
        mask = uid_mask
    else:
        raise ValueError("isfraud must be 0, 1, or 'all'.")

    out = d.loc[mask, [uid_col, "isFraud", "DT", "TransactionAmt"]].sort_values("DT")

    if out.empty:
        print(f"No rows found for uid={uid_value}, isFraud={isfraud}")
        return out

    ax = out[["DT", "TransactionAmt"]].set_index("DT").plot(
        kind="line", marker="o", figsize=(10, 4),
        title=f"{uid_col}={uid_value} | isFraud={isfraud}"
    )
    ax.set_xlabel("DT")
    ax.set_ylabel("TransactionAmt")
    plt.tight_layout()
    plt.show()

    return out


In [ ]:
# all labels for uid
# df_sub = plot_uid_timeseries(df_show, uid_value=uid, isfraud=0, start_date=START_DATE)

# only fraud rows for uid
# _ = plot_uid_timeseries(df_show, uid_value=uid, isfraud=1, start_date=START_DATE)

In [ ]:
# df_show[df_show["uid"] == uid].shape

In [ ]:
# df_sub.shape

In [ ]:
for top_uid in top_uids:
    g = df_show_filtered[df_show_filtered["uid"] == top_uid].sort_values("uid_FE")
    print(f"uid={top_uid}, n={len(g)}")
    display(g)

In [ ]:
import re

def make_Dn_columns(
    df,
    day_col="day",
    d_pattern=r"^D\d+$",
    allowed_d=(1, 2, 3, 5, 9),
    out_suffix="n",
):
    allowed = {f"D{i}" for i in allowed_d}
    d_cols = [c for c in df.columns if re.match(d_pattern, c) and c in allowed]

    out = df.copy()
    for c in d_cols:
        out[f"{c}{out_suffix}"] = out[day_col] - out[c]
    return out

# example
# g = df_show_filtered[df_show_filtered["uid"] == uid].copy()
# g = make_Dn_columns(g)

In [ ]:
X_train_copy4 = make_Dn_columns(X_train_copy4)
X_test_copy4 = make_Dn_columns(X_test_copy4)

In [ ]:
X_train_copy4.isFraud

In [ ]:
display(g)

In [ ]:
cols_show = [
    "TransactionID", "isFraud", "TransactionAmt", "card1", "card2",
    "D1n", "day", "D3n", "D3", "D5n", "D9n", "dist1", "P_emaildomain", "uid", "is_same", "D4_uid_std", "D10_uid_std", "D15_uid_std", "C13_uid_nunique", "uid_FE"
]

# If TransactionID is index, bring it back as a column first
g = g.reset_index() if "TransactionID" not in g.columns else g


display(g[[c for c in cols_show if c in g.columns]])

In [ ]:
# g[["day", "D3n", "D2n", "D5n", "D9n"]].head()

In [ ]:
df_show["isFraud"].dropna().unique().tolist()

In [ ]:
from pathlib import Path

out_dir = Path("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files")
out_dir.mkdir(parents=True, exist_ok=True)

X_train_copy4.to_pickle(out_dir / "X_train_copy4.pkl")
X_test_copy4.to_pickle(out_dir / "X_test_copy4.pkl")
y_train.to_pickle(out_dir / "y_train.pkl")

In [ ]:
X_train_copy4.isFraud

In [ ]:
any(c.endswith('_isna') for c in X_train_copy4.columns)

# TIME SERIES TRAINING

In [ ]:
cols_to_show = [
    "DT", "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday",
    "DT_M_total", "DT_W_total", "DT_D_total"
]

# keep only existing columns
cols = [c for c in cols_to_show if c in X_train_copy4.columns]
missing = [c for c in cols_to_show if c not in X_train_copy4.columns]

if missing:
    print("Missing columns:", missing)

X_train_copy4[cols].head(20)   # change 20 to any number of rows you want

In [ ]:
X_train_copy8 = X_train_copy4.copy()
X_test_copy8 = X_test_copy4.copy()

In [ ]:
X_train_copy8.drop('isFraud', axis=1, inplace=True)

In [ ]:
X_train_copy8.loc[X_train_copy8.uid == 26878, "day"]

In [ ]:
X_train_copy8.loc[X_train_copy8.uid == 26878, "DT_D"]

In [ ]:
isna_cols = [c for c in X_train_copy8.columns if c.endswith("_isna")]
X_train_copy8_isna = X_train_copy8[isna_cols].copy()

print(f"Found {len(isna_cols)} _isna columns")
display(isna_cols)

In [27]:
# Check column that remains NaN
print(X_train_copy8.isna().sum()[X_train_copy8.isna().sum() > 0])
print(X_test_copy8.isna().sum()[X_test_copy8.isna().sum() > 0])

NameError: name 'X_train_copy8' is not defined

In [ ]:
time_series_cols = [
    "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday"]

In [ ]:
encode_AG(['TransactionAmt','D4','D9','D10','D15'],time_series_cols,['mean','std'], train_df=X_train_copy8, test_df=X_test_copy8, fillna=True,usena=True)

In [ ]:
encode_AG(['C'+str(x) for x in range(1,15) if x!=3],time_series_cols,['mean'],X_train_copy8,X_test_copy8,fillna=True,usena=True)

In [ ]:
encode_AG(['M'+str(x) for x in range(1,10)],time_series_cols,['mean'], train_df=X_train_copy8, test_df=X_test_copy8, fillna=True,usena=True)

In [ ]:
encode_AG2(['P_emaildomain','dist1','id_02','cents'], time_series_cols, train_df=X_train_copy8, test_df=X_test_copy8)

In [ ]:
encode_AG(['C14'],time_series_cols,['std'],X_train_copy8,X_test_copy8,fillna=True,usena=True)

In [ ]:
encode_AG2(['C13','V314'], time_series_cols, train_df=X_train_copy8, test_df=X_test_copy8)

In [ ]:
encode_AG2(['V127','V136','V309','V307','V320'], time_series_cols, train_df=X_train_copy8, test_df=X_test_copy8)

In [ ]:
X_train_copy8['outsider15'] = (np.abs(X_train_copy8.D1-X_train_copy8.D15)>3).astype('int8')

In [ ]:
X_test_copy8['outsider15'] = (np.abs(X_test_copy8.D1-X_test_copy8.D15)>3).astype('int8')

In [ ]:
# Check column that remains NaN
print(X_train_copy8.isna().sum()[X_train_copy8.isna().sum() > 0])
print(X_test_copy8.isna().sum()[X_test_copy8.isna().sum() > 0])

In [ ]:
isna_cols = [c for c in X_train_copy8.columns if c.endswith("_isna")]
X_train_copy8_isna = X_train_copy8[isna_cols].copy()

print(f"Found {len(isna_cols)} _isna columns")
display(isna_cols)

# Feature Selection - Time Consistency
We added 28 new feature above. We have already removed 219 V Columns from correlation analysis done here. So we currently have 242 features now. We will now check each of our 242 for "time consistency". We will build 242 models. Each model will be trained on the first month of the training data and will only use one feature. We will then predict the last month of the training data. We want both training AUC and validation AUC to be above AUC = 0.5. It turns out that 19 features fail this test so we will remove them. Additionally we will remove 7 D columns that are mostly NAN. More techniques for feature selection are listed

In [ ]:
cols = list(X_train_copy8.columns)
for c in ['TransactionDT', 'D6','D7','D8','D9','D12','D13','D14']:
    if c in cols:
        cols.remove(c)

for c in ['uid', 'day', 'DT']:
    if c in cols:
        cols.remove(c)

for c in ['C3','M5','id_08','id_33']:
    if c in cols:
        cols.remove(c)

for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    if c in cols:
        cols.remove(c)

for c in [f'id_{x}' for x in range(22, 28)]:
    if c in cols:
        cols.remove(c)

In [ ]:
print('NOW USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

In [ ]:
missing_in_test = [c for c in cols if c not in X_test_copy8.columns]
print(f"Missing in X_test_copy8: {len(missing_in_test)}")
print(missing_in_test)

In [ ]:
idxT = X_train_copy8.index[:3*len(X_train_copy8)//4]
idxV = X_train_copy8.index[3*len(X_train_copy8)//4:]

In [ ]:
print(idxT)
print(idxV)

In [ ]:
# target_col = "isFraud"
# X_tr, y_tr = X_train_copy8.loc[idxT, cols], y_train[idxT]
# X_va, y_va = X_train_copy8.loc[idxV, cols], y_train[idxV]
#
# neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
# scale_pos_weight = max(1.0, neg / max(pos, 1))
#
# if BUILD95:
#     clf = xgb.XGBClassifier(
#         # n_estimators=4000,
#         # max_depth=10,
#         # learning_rate=0.03,
#         # subsample=0.8,
#         # colsample_bytree=0.5,
#         # eval_metric="auc",
#         # tree_method="hist",
#         # device="cpu",
#         # enable_categorical=True,
#         # scale_pos_weight=scale_pos_weight,
#         # early_stopping_rounds=200,
#         # random_state=42
#
#         n_estimators=2000,
#         max_depth=12,
#         learning_rate=0.02,
#         subsample=0.8,
#         colsample_bytree=0.4,
#         missing=-1,
#         eval_metric="auc",
#         tree_method="hist",      # CPU histogram algorithm
#         device="cpu",
#         early_stopping_rounds=100
#     )
#     h = clf.fit(
#         X_tr, y_tr,
#         eval_set=[(X_va, y_va)],
#         verbose=100
#     )
#     p = clf.predict_proba(X_va)[:, 1]
#     print("Best iteration:", h.best_iteration)
#     print("Best score (AUC):", h.best_score)
#     print("Validation AUC:", roc_auc_score(y_va, p))

In [ ]:
# if BUILD95:
#
#     feature_imp = pd.DataFrame(sorted(zip(clf.feature_importances_,cols)), columns=['Value','Feature'])
#     plt.figure(figsize=(20, 10))
#     sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).iloc[:50])
#     plt.title('XGB95 Most Important Features')
#     plt.tight_layout()
#     plt.show()
#     del clf, h; x=gc.collect()

In [ ]:
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
X_train_copy8['DT_M'] = X_train_copy8['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_train_copy8['DT_M'] = (X_train_copy8['DT_M'].dt.year-2017)*12 + X_train_copy8['DT_M'].dt.month

X_test_copy8['DT_M'] = X_test_copy8['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
X_test_copy8['DT_M'] = (X_test_copy8['DT_M'].dt.year-2017)*12 + X_test_copy8['DT_M'].dt.month

In [ ]:
# if BUILD95:
#     oof = np.zeros(len(X_train_copy8))
#     preds = np.zeros(len(X_test_copy8))
#
#     skf = GroupKFold(n_splits=6)
#     for i, (idxT, idxV) in enumerate( skf.split(X_train_copy8, y_train, groups=X_train_copy8['DT_M']) ):
#         month = X_train_copy8.iloc[idxV]['DT_M'].iloc[0]
#         print('Fold',i,'withholding month',month)
#         print(' rows of train =',len(idxT),'rows of holdout =',len(idxV))
#         clf = xgb.XGBClassifier(
#             n_estimators=5000,
#             max_depth=12,
#             learning_rate=0.02,
#             subsample=0.8,
#             colsample_bytree=0.4,
#             missing=-1,
#             eval_metric='auc',
#             # USE CPU
#             #nthread=4,
#             tree_method='hist',
#             device = 'cpu',
#             # USE GPU
#             # tree_method='gpu_hist',
#             early_stopping_rounds=100
#         )
#         h = clf.fit(X_train_copy8[cols].iloc[idxT], y_train.iloc[idxT],
#                 eval_set=[(X_train_copy8[cols].iloc[idxV],y_train.iloc[idxV])],
#                 verbose=100)
#
#         # Fold best iteration + best score
#         try:
#             print(f"Fold {i} best_iteration={clf.best_iteration}, best_auc={clf.best_score:.6f}")
#         except Exception:
#             # fallback for versions where best_score is not exposed
#             ev = clf.evals_result()["validation_0"]["auc"]
#             best_iter = int(np.argmax(ev))
#             best_auc = float(np.max(ev))
#             print(f"Fold {i} best_iteration={best_iter}, best_auc={best_auc:.6f}")
#
#         oof[idxV] += clf.predict_proba(X_train_copy8[cols].iloc[idxV])[:,1]
#         preds += clf.predict_proba(X_test_copy8[cols])[:,1]/skf.n_splits
#         del h, clf
#         x=gc.collect()
#     print('#'*20)
#     print ('XGB95 OOF CV=',roc_auc_score(y_train,oof))

In [ ]:
# if BUILD95:
#     plt.hist(oof,bins=100)
#     plt.ylim((0,5000))
#     plt.title('XGB OOF')
#     plt.show()
#
#     X_train_copy8['oof'] = oof
#     X_train_copy8.reset_index(inplace=True)
#     X_train_copy8[['TransactionID','oof']].to_csv('oof_xgb_95.csv')
#     X_train_copy8.set_index('TransactionID',drop=True,inplace=True)
#
# else: X_train_copy8['oof'] = 0

# TRAIN WITHOUT V COLUMNS

In [ ]:
non_v_cols_train = [c for c in X_train_copy8.columns if not c.startswith('V')]
non_v_cols_test  = [c for c in X_test_copy8.columns  if not c.startswith('V')]

X_train_copy9 = X_train_copy8[non_v_cols_train].copy()
X_test_copy9  = X_test_copy8[non_v_cols_test].copy()

In [ ]:
cols = list(X_train_copy9.columns)
for c in ['TransactionDT', 'D6','D7','D8','D9','D12','D13','D14']:
    if c in cols:
        cols.remove(c)

for c in ['uid', 'day', 'oof', 'DT']:
    if c in cols:
        cols.remove(c)

for c in ['C3','M5','id_08','id_33']:
    if c in cols:
        cols.remove(c)

for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    if c in cols:
        cols.remove(c)

for c in [f'id_{x}' for x in range(22, 28)]:
    if c in cols:
        cols.remove(c)

In [ ]:
print('NOW USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

In [ ]:
v_cols = [c for c in X_train_copy8.columns if c.startswith('V')]
len(v_cols)

In [ ]:
idxT = X_train_copy9.index[:3*len(X_train_copy9)//4]
idxV = X_train_copy9.index[3*len(X_train_copy9)//4:]

In [ ]:
# if BUILD96:
#     clf = xgb.XGBClassifier(
#         n_estimators=2000,
#         max_depth=12,
#         learning_rate=0.02,
#         subsample=0.8,
#         colsample_bytree=0.4,
#         missing=-1,
#         eval_metric='auc',
#         #nthread=4,
#         tree_method='hist',
#         device = 'cpu',
#         # USE GPU
#         # tree_method='gpu_hist',
#         early_stopping_rounds=100
#     )
#     h = clf.fit(X_train_copy9.loc[idxT,cols], y_train[idxT],
#         eval_set=[(X_train_copy9.loc[idxV,cols],y_train[idxV])],
#         verbose=50)
#     print("Best iteration:", h.best_iteration)
#     print("Best score:", h.best_score)

In [ ]:
# if BUILD96:
#
#     feature_imp = pd.DataFrame(sorted(zip(clf.feature_importances_,cols)), columns=['Value','Feature'])
#
#     plt.figure(figsize=(20, 10))
#     sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).iloc[:50])
#     plt.title('XGB96 Most Important')
#     plt.tight_layout()
#     plt.show()
#
#     del clf, h; x=gc.collect()

In [ ]:
# if BUILD96:
#     oof = np.zeros(len(X_train_copy9))
#     preds = np.zeros(len(X_test_copy9))
#
#     skf = GroupKFold(n_splits=6)
#     for i, (idxT, idxV) in enumerate( skf.split(X_train_copy9, y_train, groups=X_train_copy9['DT_M']) ):
#         month = X_train_copy9.iloc[idxV]['DT_M'].iloc[0]
#         print('Fold',i,'withholding month',month)
#         print(' rows of train =',len(idxT),'rows of holdout =',len(idxV))
#         clf = xgb.XGBClassifier(
#             n_estimators=5000,
#             max_depth=12,
#             learning_rate=0.02,
#             subsample=0.8,
#             colsample_bytree=0.4,
#             missing=-1,
#             eval_metric='auc',
#             # USE CPU
#             #nthread=4,
#             tree_method='hist',
#             # USE GPU
#             device = 'cpu',
#             # USE GPU
#             # tree_method='gpu_hist',
#             early_stopping_rounds=100
#         )
#         h = clf.fit(X_train_copy9[cols].iloc[idxT], y_train.iloc[idxT],
#                 eval_set=[(X_train_copy9[cols].iloc[idxV],y_train.iloc[idxV])],
#                 verbose=100)
#
#         # Fold best iteration + best score
#         try:
#             print(f"Fold {i} best_iteration={clf.best_iteration}, best_auc={clf.best_score:.6f}")
#         except Exception:
#             # fallback for versions where best_score is not exposed
#             ev = clf.evals_result()["validation_0"]["auc"]
#             best_iter = int(np.argmax(ev))
#             best_auc = float(np.max(ev))
#             print(f"Fold {i} best_iteration={best_iter}, best_auc={best_auc:.6f}")
#
#         oof[idxV] += clf.predict_proba(X_train_copy9[cols].iloc[idxV])[:,1]
#         preds += clf.predict_proba(X_test_copy9[cols])[:,1]/skf.n_splits
#         del h, clf
#         x=gc.collect()
#     print('#'*20)
#     print ('XGB96 OOF CV=',roc_auc_score(y_train,oof))

In [ ]:
# if BUILD96:
#     plt.hist(oof,bins=100)
#     plt.ylim((0,5000))
#     plt.title('XGB OOF')
#     plt.show()
#
#     X_train_copy9['oof'] = oof
#     X_train_copy9.reset_index(inplace=True)
#     X_train_copy9[['TransactionID','oof']].to_csv('oof_xgb_96.csv')
#     X_train_copy9.set_index('TransactionID',drop=True,inplace=True)

# TRAIN WITH UID COMBINING WITH TIME_SERIES_COLS

In [ ]:
X_train_copy10 = X_train_copy4.copy()
X_test_copy10 = X_test_copy4.copy()

In [ ]:
X_train_copy10.drop('isFraud', axis=1, inplace=True)

In [ ]:
print(X_train_copy10.isna().sum()[X_train_copy10.isna().sum() > 0])
print(X_test_copy10.isna().sum()[X_test_copy10.isna().sum() > 0])

In [ ]:
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_M", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_W", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_D", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_hour", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_day_week", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_day_month", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "DT_week_month", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "is_december", X_train_copy10, X_test_copy10)
X_train_copy10, X_test_copy10 = encode_CB("uid", "is_holiday", X_train_copy10, X_test_copy10)

In [ ]:
uid_time_series = ['uid_DT_M' , 'uid_DT_W' , 'uid_DT_D' , 'uid_DT_hour' , 'uid_DT_day_week' , 'uid_DT_day_month' , 'uid_DT_week_month' , 'uid_is_december' , 'uid_is_holiday']

In [ ]:
X_train_copy10.uid_FE

In [ ]:
encode_AG(['TransactionAmt','D4','D9','D10','D15'], uid_time_series,['mean','std'], train_df=X_train_copy10, test_df=X_test_copy10, fillna=True,usena=True)

In [ ]:
encode_AG(['C'+str(x) for x in range(1,15) if x!=3],uid_time_series,['mean'],X_train_copy10,X_test_copy10,fillna=True,usena=True)

In [ ]:
encode_AG(['M'+str(x) for x in range(1,10)],uid_time_series,['mean'], train_df=X_train_copy10, test_df=X_test_copy10, fillna=True,usena=True)

In [ ]:
encode_AG2(['P_emaildomain','dist1','id_02','cents'], uid_time_series, train_df=X_train_copy10, test_df=X_test_copy10)

In [ ]:
encode_AG(['C14'],uid_time_series,['std'],X_train_copy10,X_test_copy10,fillna=True,usena=True)

In [ ]:
encode_AG2(['C13','V314'], uid_time_series, train_df=X_train_copy10, test_df=X_test_copy10)

In [ ]:
encode_AG2(['V127','V136','V309','V307','V320'], uid_time_series, train_df=X_train_copy10, test_df=X_test_copy10)

In [ ]:
X_train_copy10['outsider15'] = (np.abs(X_train_copy10.D1-X_train_copy10.D15)>3).astype('int8')

In [ ]:
X_test_copy10['outsider15'] = (np.abs(X_test_copy10.D1-X_test_copy10.D15)>3).astype('int8')

In [ ]:
cols = list(X_train_copy10.columns)
for c in ['TransactionDT', 'D6','D7','D8','D9','D12','D13','D14']:
    if c in cols:
        cols.remove(c)

for c in ['uid', 'day', 'oof', 'DT']:
    if c in cols:
        cols.remove(c)

for c in ['C3','M5','id_08','id_33']:
    if c in cols:
        cols.remove(c)

for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    if c in cols:
        cols.remove(c)

for c in [f'id_{x}' for x in range(22, 28)]:
    if c in cols:
        cols.remove(c)

In [ ]:
print('NOW USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

In [ ]:
idxT = X_train_copy10.index[:3*len(X_train_copy10)//4]
idxV = X_train_copy10.index[3*len(X_train_copy10)//4:]

In [28]:
# if BUILD95:
#     clf = xgb.XGBClassifier(
#         n_estimators=2000,
#         max_depth=12,
#         learning_rate=0.02,
#         subsample=0.8,
#         colsample_bytree=0.4,
#         missing=-1,
#         eval_metric='auc',
#         #nthread=4,
#         tree_method='hist',
#         device = 'cpu',
#         # USE GPU
#         # tree_method='gpu_hist',
#         early_stopping_rounds=100
#     )
#     h = clf.fit(X_train_copy10.loc[idxT,cols], y_train[idxT],
#         eval_set=[(X_train_copy10.loc[idxV,cols],y_train[idxV])],
#         verbose=50)
#     print("Best iteration:", h.best_iteration)
#     print("Best score:", h.best_score)

In [29]:
# if BUILD95:
#
#     feature_imp = pd.DataFrame(sorted(zip(clf.feature_importances_,cols)), columns=['Value','Feature'])
#
#     plt.figure(figsize=(20, 10))
#     sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).iloc[:50])
#     plt.title('XGB96 Most Important')
#     plt.tight_layout()
#     plt.show()
#
#     del clf, h; x=gc.collect()

In [30]:
# if BUILD95:
#     oof = np.zeros(len(X_train_copy10))
#     preds = np.zeros(len(X_test_copy10))
#
#     skf = GroupKFold(n_splits=6)
#     for i, (idxT, idxV) in enumerate( skf.split(X_train_copy10, y_train, groups=X_train_copy10['DT_M']) ):
#         month = X_train_copy10.iloc[idxV]['DT_M'].iloc[0]
#         print('Fold',i,'withholding month',month)
#         print(' rows of train =',len(idxT),'rows of holdout =',len(idxV))
#         clf = xgb.XGBClassifier(
#             n_estimators=5000,
#             max_depth=12,
#             learning_rate=0.02,
#             subsample=0.8,
#             colsample_bytree=0.4,
#             missing=-1,
#             eval_metric='auc',
#             # USE CPU
#             #nthread=4,
#             tree_method='hist',
#             # USE GPU
#             device = 'cpu',
#             # USE GPU
#             # tree_method='gpu_hist',
#             early_stopping_rounds=100
#         )
#         h = clf.fit(X_train_copy10[cols].iloc[idxT], y_train.iloc[idxT],
#                 eval_set=[(X_train_copy10[cols].iloc[idxV],y_train.iloc[idxV])],
#                 verbose=100)
#
#         # Fold best iteration + best score
#         try:
#             print(f"Fold {i} best_iteration={clf.best_iteration}, best_auc={clf.best_score:.6f}")
#         except Exception:
#             # fallback for versions where best_score is not exposed
#             ev = clf.evals_result()["validation_0"]["auc"]
#             best_iter = int(np.argmax(ev))
#             best_auc = float(np.max(ev))
#             print(f"Fold {i} best_iteration={best_iter}, best_auc={best_auc:.6f}")
#
#         oof[idxV] += clf.predict_proba(X_train_copy10[cols].iloc[idxV])[:,1]
#         preds += clf.predict_proba(X_test_copy10[cols])[:,1]/skf.n_splits
#         del h, clf
#         x=gc.collect()
#     print('#'*20)
#     print ('XGB96 OOF CV=',roc_auc_score(y_train,oof))

In [31]:
# if BUILD95:
#     plt.hist(oof,bins=100)
#     plt.ylim((0,5000))
#     plt.title('XGB OOF')
#     plt.show()
#
#     X_train_copy10['oof'] = oof
#     X_train_copy10.reset_index(inplace=True)
#     X_train_copy10[['TransactionID','oof']].to_csv('oof_xgb_96.csv')
#     X_train_copy10.set_index('TransactionID',drop=True,inplace=True)

# TRAIN WITHOUT V COLUMNS WITH UID COMBINING WITH TIME SERIES COLUMNS

In [32]:
non_v_cols_train = [c for c in X_train_copy10.columns if not c.startswith('V')]
non_v_cols_test  = [c for c in X_test_copy10.columns  if not c.startswith('V')]

X_train_copy11 = X_train_copy10[non_v_cols_train].copy()
X_test_copy11  = X_test_copy10[non_v_cols_test].copy()

NameError: name 'X_train_copy10' is not defined

In [ ]:
cols = list(X_train_copy11.columns)
for c in ['TransactionDT', 'D6','D7','D8','D9','D12','D13','D14']:
    if c in cols:
        cols.remove(c)

for c in ['uid', 'day', 'oof', 'DT']:
    if c in cols:
        cols.remove(c)

for c in ['C3','M5','id_08','id_33']:
    if c in cols:
        cols.remove(c)

for c in ['card4','id_07','id_14','id_21','id_30','id_32','id_34']:
    if c in cols:
        cols.remove(c)

for c in [f'id_{x}' for x in range(22, 28)]:
    if c in cols:
        cols.remove(c)

In [ ]:
print('NOW USING THE FOLLOWING',len(cols),'FEATURES.')
np.array(cols)

In [ ]:
idxT = X_train_copy11.index[:3*len(X_train_copy11)//4]
idxV = X_train_copy11.index[3*len(X_train_copy11)//4:]

In [ ]:
# if BUILD96:
#     clf = xgb.XGBClassifier(
#         n_estimators=2000,
#         max_depth=12,
#         learning_rate=0.02,
#         subsample=0.8,
#         colsample_bytree=0.4,
#         missing=-1,
#         eval_metric='auc',
#         #nthread=4,
#         tree_method='hist',
#         device = 'cpu',
#         # USE GPU
#         # tree_method='gpu_hist',
#         early_stopping_rounds=100
#     )
#     h = clf.fit(X_train_copy11.loc[idxT,cols], y_train[idxT],
#         eval_set=[(X_train_copy11.loc[idxV,cols],y_train[idxV])],
#         verbose=50)
#     print("Best iteration:", h.best_iteration)
#     print("Best score:", h.best_score)

In [ ]:
# if BUILD96:
#
#     feature_imp = pd.DataFrame(sorted(zip(clf.feature_importances_,cols)), columns=['Value','Feature'])
#
#     plt.figure(figsize=(20, 10))
#     sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value", ascending=False).iloc[:50])
#     plt.title('XGB96 Most Important')
#     plt.tight_layout()
#     plt.show()
#
#     del clf, h; x=gc.collect()

In [ ]:
# if BUILD96:
#     oof = np.zeros(len(X_train_copy11))
#     preds = np.zeros(len(X_test_copy11))
#
#     skf = GroupKFold(n_splits=6)
#     for i, (idxT, idxV) in enumerate( skf.split(X_train_copy11, y_train, groups=X_train_copy11['DT_M']) ):
#         month = X_train_copy11.iloc[idxV]['DT_M'].iloc[0]
#         print('Fold',i,'withholding month',month)
#         print(' rows of train =',len(idxT),'rows of holdout =',len(idxV))
#         clf = xgb.XGBClassifier(
#             n_estimators=5000,
#             max_depth=12,
#             learning_rate=0.02,
#             subsample=0.8,
#             colsample_bytree=0.4,
#             missing=-1,
#             eval_metric='auc',
#             # USE CPU
#             #nthread=4,
#             tree_method='hist',
#             # USE GPU
#             device = 'cpu',
#             # USE GPU
#             # tree_method='gpu_hist',
#             early_stopping_rounds=100
#         )
#         h = clf.fit(X_train_copy11[cols].iloc[idxT], y_train.iloc[idxT],
#                 eval_set=[(X_train_copy11[cols].iloc[idxV],y_train.iloc[idxV])],
#                 verbose=100)
#
#         # Fold best iteration + best score
#         try:
#             print(f"Fold {i} best_iteration={clf.best_iteration}, best_auc={clf.best_score:.6f}")
#         except Exception:
#             # fallback for versions where best_score is not exposed
#             ev = clf.evals_result()["validation_0"]["auc"]
#             best_iter = int(np.argmax(ev))
#             best_auc = float(np.max(ev))
#             print(f"Fold {i} best_iteration={best_iter}, best_auc={best_auc:.6f}")
#
#         oof[idxV] += clf.predict_proba(X_train_copy11[cols].iloc[idxV])[:,1]
#         preds += clf.predict_proba(X_test_copy11[cols])[:,1]/skf.n_splits
#         del h, clf
#         x=gc.collect()
#     print('#'*20)
#     print ('XGB96 OOF CV=',roc_auc_score(y_train,oof))

In [ ]:
# if BUILD96:
#     plt.hist(oof,bins=100)
#     plt.ylim((0,5000))
#     plt.title('XGB OOF')
#     plt.show()
#
#     X_train_copy11['oof'] = oof
#     X_train_copy11.reset_index(inplace=True)
#     X_train_copy11[['TransactionID','oof']].to_csv('oof_xgb_96.csv')
#     X_train_copy11.set_index('TransactionID',drop=True,inplace=True)